<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/BIGLUST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BigLust (SDXL)

1. **Runtime → Change runtime type → T4 GPU.**
2. Run **cell 1** (install + downloads; allow Google Drive access), **cell 2** (saves the server program) and
   **cell 3** (starts it — prints the `READY:` URL after BigLust loads).
3. Paste the URL into the website's **Backend URL** field. Add more SDXL checkpoints and LoRAs from the website,
   use the presets, and turn on **Face fix** for sharper faces that look more like a person LoRA.

The server runs as a separate program: if a generation runs out of memory, only the server restarts (same URL,
the model reloads in about a minute) and this notebook keeps running. **Cell 4** shows the server's log and memory use.

**Face reference:** cell 1 also installs IP-Adapter FaceID Plus v2. On the website, add 1–3 pictures of a character's face under *Face reference* and every image keeps that face — handy for comics and short films. Three choices with the switch there: **FaceID** (the original), **Both** (stacked, each with its own strength) and **PuLID** (keeps the face but leaves the lighting and style alone, so it can be turned up further — ~0.6–1.0 works well; its models download the first time you use it). Only use pictures of real people who have agreed to it.

In [ ]:
# Cell 1: install ComfyUI and download BigLust
%cd /content
# --- Google Drive: keep downloaded LoRAs between Colab sessions (set to False to skip) ---
USE_GOOGLE_DRIVE = True
def mount_google_drive():
    """Connects Google Drive. If that fails, the notebook carries on WITHOUT Drive instead of stopping
    (downloaded LoRAs then stay on this runtime only), so a Drive problem can never block you."""
    import os, shutil, time
    from google.colab import drive
    mp = "/content/drive"
    if os.path.isdir(mp) and not os.path.ismount(mp) and os.listdir(mp):
        # files left by an earlier failed run block the mount ("Mountpoint must not already contain files")
        keep = f"/content/drive_leftovers_{int(time.time())}"
        shutil.move(mp, keep)
        print(f"Moved files that were in Google Drive's way to {keep}")
    error = None
    for attempt in (1, 2):   # a second try after a clean unmount fixes most "mount failed" errors
        try:
            drive.mount(mp, force_remount=True)  # Colab asks you to allow access
            error = None
            break
        except Exception as e:
            error = e
            print(f"Google Drive try {attempt} failed ({e})" + (" — trying once more..." if attempt == 1 else ""))
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            if os.path.isdir(mp) and not os.path.ismount(mp) and os.listdir(mp):
                shutil.move(mp, f"/content/drive_leftovers_{int(time.time())}")
            time.sleep(3)
    if error is not None:
        e = error
        print(f"\n⚠️  Google Drive did not connect: {e}")
        print("    Carrying on WITHOUT Google Drive: LoRAs you download stay on this runtime only.")
        print("    To fix it next time: Runtime → Disconnect and delete runtime, run again, sign in with the SAME")
        print("    Google account as Colab, and on Google's page tick EVERY box (especially 'See, edit, create and")
        print("    delete all of your Google Drive files'). Allow pop-ups / third-party cookies for colab.research.google.com.\n")
        return False
    if not os.path.isdir(mp + "/MyDrive"):
        print("⚠️  Google Drive connected, but MyDrive is missing — carrying on without it.")
        return False
    return True
if USE_GOOGLE_DRIVE and not mount_google_drive():
    USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    pass  # Drive was connected above (mount_google_drive)

!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!git clone https://github.com/city96/ComfyUI-GGUF /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q torchsde gguf
!pip install -q -r /content/ComfyUI/requirements.txt
!apt install aria2 -qqy

!mkdir -p /content/ComfyUI/models/checkpoints /content/ComfyUI/models/loras

# Store LoRAs in MyDrive/ComfyUI/loras_sdxl: ComfyUI's loras folder becomes a link to it, so LoRAs
# downloaded from the website are saved to Drive and show up again in every new session.
import os, shutil
LORA_LOCAL = "/content/ComfyUI/models/loras"
if USE_GOOGLE_DRIVE:
    DRIVE_LORAS = "/content/drive/MyDrive/ComfyUI/loras_sdxl"
    os.makedirs(DRIVE_LORAS, exist_ok=True)
    if not os.path.islink(LORA_LOCAL):
        if os.path.isdir(LORA_LOCAL):
            for f in os.listdir(LORA_LOCAL):  # keep anything already downloaded in this session
                src, dst = os.path.join(LORA_LOCAL, f), os.path.join(DRIVE_LORAS, f)
                if not f.startswith("put_") and not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(LORA_LOCAL)
        os.symlink(DRIVE_LORAS, LORA_LOCAL)
    n = len([f for f in os.listdir(DRIVE_LORAS) if not f.endswith(".part")])
    print(f"LoRAs are stored in Google Drive (MyDrive/ComfyUI/loras_sdxl): {n} file(s)")
else:
    if os.path.islink(LORA_LOCAL):   # still linked to Drive from an earlier run
        os.unlink(LORA_LOCAL)
    os.makedirs(LORA_LOCAL, exist_ok=True)
    print("LoRAs are stored on this runtime only (lost when the runtime is deleted)")

# Download BigLust (SDXL checkpoint)
!wget -q --show-progress -O /content/ComfyUI/models/checkpoints/biglust.safetensors \
  "https://civitai.com/api/download/models/1081768?fileId=986850&token=777a5632707c15ecf54e13eb3b1b862b"

# Download VAE (SDXL VAE is recommended for BigLust)
!mkdir -p /content/ComfyUI/models/vae
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors \
  -d /content/ComfyUI/models/vae -o sdxl_vae.safetensors


# Face fix (website toggle): the face detector ADetailer uses
!pip install -q ultralytics
!mkdir -p /content/ComfyUI/models/ultralytics
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt \
  -d /content/ComfyUI/models/ultralytics -o face_yolov8m.pt

# Face reference (website): IP-Adapter FaceID Plus v2 keeps a character's face from 1-3 pictures
!pip install -q insightface==0.7.3 onnxruntime
!test -d /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus || git clone -q https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
!cd /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus && git checkout -q a0f451a
!mkdir -p /content/ComfyUI/models/ipadapter /content/ComfyUI/models/clip_vision /content/ComfyUI/models/insightface
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin \
  -d /content/ComfyUI/models/ipadapter -o ip-adapter-faceid-plusv2_sdxl.bin
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors \
  -d /content/ComfyUI/models/ipadapter -o ip-adapter-faceid-plusv2_sdxl_lora.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors \
  -d /content/ComfyUI/models/clip_vision -o CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors
# the face recogniser (InsightFace buffalo_l) downloads itself on first use; do it now and check it works
!python -c "from insightface.app import FaceAnalysis; FaceAnalysis(name='buffalo_l', root='/content/ComfyUI/models/insightface', providers=['CPUExecutionProvider']); print('Face reference ready.')" 2>&1 | tail -1

# Face reference, PuLID method (website toggle next to FaceID): the code and the libraries it needs. Its models
# (PuLID, the antelopev2 face recogniser, EVA-CLIP: ~2 GB) download the first time you pick PuLID, not here.
!test -d /content/ComfyUI/custom_nodes/PuLID_ComfyUI || git clone -q https://github.com/cubiq/PuLID_ComfyUI /content/ComfyUI/custom_nodes/PuLID_ComfyUI
!cd /content/ComfyUI/custom_nodes/PuLID_ComfyUI && git checkout -q 93e0c4c
!pip install -q facexlib ftfy timm

# Pose / depth / style picture (website, next to Face reference): the pose and depth readers. The ControlNets and
# IP-Adapter Plus themselves download the first time each mode is used (2.5 GB / 2.5 GB / 0.85 GB), not here.
!pip install -q controlnet_aux==0.0.10
!mkdir -p /content/ComfyUI/models/controlnet

# Check downloads
!ls -lh /content/ComfyUI/models/checkpoints/
!ls -lh /content/ComfyUI/models/loras/
!ls -lh /content/ComfyUI/models/vae/

In [ ]:
%%writefile /content/biglust_server.py
# Cell 2: the generation server, saved as its own program (/content/biglust_server.py).
# Cell 3 runs it separately from this notebook: if a generation runs out of memory only this program
# dies, and cell 3 restarts it automatically (same URL) instead of the whole Colab session crashing.
# Running this cell only saves the file; edit it here and re-run cells 2 and 3 to apply changes.
# Add any SDXL (or SD1.5) checkpoint or LoRA from the website with "+ Download Checkpoint" / "+ Download LoRA".
import os
import sys

os.chdir("/content/ComfyUI")
sys.path.insert(0, "/content/ComfyUI")

CRASH_FILE = "/content/server_crash.txt"


def last_crash_message():
    """Set by cell 3 when the server crashed and was restarted, so the website can say why a job vanished."""
    try:
        return open(CRASH_FILE).read().strip()
    except OSError:
        return ""


print("Loading BigLust...", flush=True)
import torch
import random, time
from PIL import Image
import numpy as np
import logging
import warnings

# Hide warnings
logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from nodes import NODE_CLASS_MAPPINGS

CheckpointLoaderSimple = NODE_CLASS_MAPPINGS["CheckpointLoaderSimple"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

with torch.inference_mode():
    # Load BigLust checkpoint
    ckpt = CheckpointLoaderSimple.load_checkpoint("biglust.safetensors")
    unet = ckpt[0]
    clip = ckpt[1]
    vae = ckpt[2]

print("BigLust loaded.", flush=True)


# ---------------------------------------------------------------- API for the website
import os
import re
import gc
import time
import uuid
import base64
import random
import threading
import subprocess
import traceback
import urllib.parse
from io import BytesIO
from typing import List, Optional

import numpy as np
import requests
import torch
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

import comfy.utils
import comfy.model_management
from nodes import NODE_CLASS_MAPPINGS

# Optional: your Civitai API key (some models need it). The website also sends one.
CIVITAI_TOKEN = os.environ.get("CIVITAI_TOKEN", "")

MODELS_DIR = "/content/ComfyUI/models"
CKPT_DIR = os.path.join(MODELS_DIR, "checkpoints")
LORA_DIR = os.path.join(MODELS_DIR, "loras")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)
MODEL_EXTS = (".safetensors", ".ckpt", ".pt", ".pth", ".bin")
DEFAULT_CHECKPOINT = "biglust.safetensors"

CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
CheckpointLoaderSimple = NODE_CLASS_MAPPINGS["CheckpointLoaderSimple"]()

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ---------------- checkpoints: keep exactly one loaded ----------------
loaded = globals().get("loaded") if isinstance(globals().get("loaded"), dict) else {"name": None, "model": None, "clip": None, "vae": None}
if loaded["name"] is None and "unet" in globals() and "clip" in globals() and "vae" in globals():
    loaded.update(name=DEFAULT_CHECKPOINT, model=unet, clip=clip, vae=vae)  # loaded by cell 3


def list_files(folder):
    out = []
    for root, dirs, files in os.walk(folder):
        # skip training save points (e.g. a copied output folder: <name>-step00000200-state/model.safetensors)
        dirs[:] = [d for d in dirs if not d.endswith("-state") and not d.startswith(".")]
        for f in files:
            if f.lower().endswith(MODEL_EXTS) and f.lower() not in ("optimizer.bin", "scheduler.bin", "model.safetensors"):
                out.append(os.path.relpath(os.path.join(root, f), folder))
    return sorted(out, key=str.lower)


def get_checkpoint(name):
    name = name or DEFAULT_CHECKPOINT
    if loaded["name"] == name:
        return loaded["model"], loaded["clip"], loaded["vae"]
    if name not in list_files(CKPT_DIR):
        raise ValueError(f"Checkpoint not found: {name}")
    print(f"Switching checkpoint: {loaded['name']} -> {name}")
    # free the previous checkpoint first so two SDXL models never sit in RAM together
    loaded.update(name=None, model=None, clip=None, vae=None)
    for g in ("unet", "clip", "vae"):
        globals().pop(g, None)
    comfy.model_management.unload_all_models()
    gc.collect()
    comfy.model_management.soft_empty_cache()
    with torch.inference_mode():
        m, c, v = CheckpointLoaderSimple.load_checkpoint(name)[:3]
    loaded.update(name=name, model=m, clip=c, vae=v)
    return m, c, v


# ---------------- downloads (background, streamed to disk) ----------------
JOBS = globals().get("JOBS") if isinstance(globals().get("JOBS"), dict) else {}  # kept across re-runs


def new_job(kind, **extra):
    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {"kind": kind, "status": "queued", "created": time.time(), "error": None, **extra}
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 1800]:
        JOBS.pop(jid, None)
    return job_id


def resolve_url(url):
    """Turn page links into direct download links. Returns (download_url, suggested_name)."""
    url = url.strip()
    parsed = urllib.parse.urlparse(url)
    qs = urllib.parse.parse_qs(parsed.query)
    token = (qs.get("token") or [CIVITAI_TOKEN])[0]
    host = parsed.netloc.lower()

    if "civitai.com" in host or "civitai.red" in host:
        api_headers = {"Authorization": f"Bearer {token}"} if token else {}
        m = re.search(r"/api/download/models/(\d+)", parsed.path)
        if not m:
            version_id = (qs.get("modelVersionId") or [None])[0]
            if not version_id:
                mm = re.search(r"/models/(\d+)", parsed.path)
                if not mm:
                    raise ValueError("Unrecognised Civitai link. Use a model page or download link.")
                info = requests.get(f"https://civitai.com/api/v1/models/{mm.group(1)}", headers=api_headers, timeout=30)
                info.raise_for_status()
                versions = info.json().get("modelVersions") or []
                if not versions:
                    raise ValueError("That Civitai model has no downloadable versions.")
                version_id = versions[0]["id"]  # newest version
            url = f"https://civitai.com/api/download/models/{version_id}"
        else:
            url = f"https://civitai.com/api/download/models/{m.group(1)}"
        if token:
            url += "?token=" + urllib.parse.quote(token)
        return url, None

    if "huggingface.co" in host:
        url = url.replace("/blob/", "/resolve/")
        return url, os.path.basename(urllib.parse.urlparse(url).path)

    return url, os.path.basename(parsed.path)


def filename_from_response(resp, fallback):
    cd = resp.headers.get("content-disposition", "")
    m = re.search(r"filename\*=(?:UTF-8'')?([^;]+)", cd, re.I) or re.search(r'filename="?([^";]+)"?', cd, re.I)
    name = urllib.parse.unquote(m.group(1).strip()) if m else (fallback or "")
    name = os.path.basename(name).strip().strip('"')
    name = re.sub(r"[^\w.\-+() ]", "_", name)
    return name


def run_download(job_id, url, kind):
    job = JOBS[job_id]
    job["status"] = "running"
    folder = LORA_DIR if kind == "lora" else CKPT_DIR
    tmp_path = None
    try:
        dl_url, suggested = resolve_url(url)
        with requests.get(dl_url, stream=True, timeout=60, allow_redirects=True,
                          headers={"User-Agent": "Mozilla/5.0"}) as r:
            if r.status_code in (401, 403):
                raise ValueError("Access denied — this model needs a Civitai API key (set CIVITAI_TOKEN or add ?token=...)")
            r.raise_for_status()
            if "text/html" in r.headers.get("content-type", ""):
                raise ValueError("The link returned a web page, not a model file. Use the download link.")
            name = filename_from_response(r, suggested)
            if not name.lower().endswith(MODEL_EXTS):
                name = f"{kind}_{int(time.time())}.safetensors"
            job["filename"] = name
            path = os.path.join(folder, name)
            if os.path.exists(path):
                job["status"], job["progress"] = "done", 100
                return
            total = int(r.headers.get("content-length") or 0)
            job["total_mb"] = round(total / 1e6) if total else None
            tmp_path = path + ".part"
            done = 0
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)
                    done += len(chunk)
                    job["done_mb"] = round(done / 1e6)
                    job["progress"] = round(done * 100 / total) if total else None
            os.replace(tmp_path, path)
            tmp_path = None
        job["status"], job["progress"] = "done", 100
        print(f"Downloaded {kind}: {name} ({job.get('done_mb')} MB)")
    except Exception as e:
        traceback.print_exc()
        job["status"], job["error"] = "error", str(e)
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)


class ModelDownload(BaseModel):
    url: str
    type: str = "checkpoint"


@app.post("/download_model")
def download_model(req: ModelDownload):
    kind = "lora" if req.type == "lora" else "checkpoint"
    job_id = new_job("download", type=kind, filename=None, progress=0, done_mb=0, total_mb=None)
    threading.Thread(target=run_download, args=(job_id, req.url, kind), daemon=True).start()
    return {"job_id": job_id, "status": "queued", "type": kind}


# ---------------- LoRA tags (the website's prompt enhancer) ----------------
# kohya sd-scripts (the LoRA Training notebook, and most Civitai LoRAs) stores how often every caption tag was used in
# training inside the .safetensors file ("ss_tag_frequency"). Reading the file's small JSON header is instant — the
# LoRA itself is not loaded. The website uses this to suggest the trigger word and the LoRA's own tags.
_lora_tags_cache = {}


def read_lora_tags(path):
    import json
    import struct
    with open(path, "rb") as fh:
        n = struct.unpack("<Q", fh.read(8))[0]
        if n > 100_000_000:
            raise ValueError("not a .safetensors file")
        meta = (json.loads(fh.read(n)) or {}).get("__metadata__") or {}

    def as_json(key):
        try:
            return json.loads(meta.get(key) or "{}")
        except Exception:
            return {}
    counts, spelling = {}, {}
    for tags in as_json("ss_tag_frequency").values():
        for tag, count in (tags or {}).items():
            tag = str(tag).strip()
            if not tag:
                continue
            key = tag.lower()
            spelling.setdefault(key, tag)
            counts[key] = counts.get(key, 0) + int(count or 0)
    images = sum(int((d or {}).get("img_count") or 0) for d in as_json("ss_dataset_dirs").values())
    if not images:
        try:
            images = int(meta.get("ss_num_train_images") or 0)
        except ValueError:
            images = 0
    images = max(images, max(counts.values(), default=0))
    trigger = [t.strip() for t in re.split(r"[,\n]", meta.get("modelspec.trigger_phrase") or meta.get("ss_trigger_words") or "") if t.strip()]
    tags = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:400]
    return {"images": images, "trigger": trigger, "tags": [{"tag": spelling[k], "count": c} for k, c in tags],
            "base_model": meta.get("ss_sd_model_name") or meta.get("modelspec.architecture") or None}


@app.get("/lora_tags")
def lora_tags(name: str):
    path = os.path.realpath(os.path.join(LORA_DIR, name))
    root = os.path.realpath(LORA_DIR)
    if not (path == root or path.startswith(root + os.sep)) or not os.path.isfile(path):
        return {"error": f"LoRA not found on the server: {name}"}
    if not path.lower().endswith(".safetensors"):
        return {"name": name, "images": 0, "trigger": [], "tags": [], "note": "Only .safetensors LoRAs carry their training tags"}
    key = (path, os.path.getmtime(path))
    if key not in _lora_tags_cache:
        try:
            _lora_tags_cache[key] = read_lora_tags(path)
        except Exception as e:
            return {"error": f"Could not read {name}: {e}"}
    out = {"name": name, **_lora_tags_cache[key]}
    if not out["tags"]:
        out["note"] = "This LoRA has no training tags saved in it (made without kohya, or with metadata turned off)"
    return out


@app.get("/user_models")
def get_user_models():
    ckpts = list_files(CKPT_DIR)
    if DEFAULT_CHECKPOINT in ckpts:  # default first
        ckpts.remove(DEFAULT_CHECKPOINT)
        ckpts.insert(0, DEFAULT_CHECKPOINT)
    return {
        "checkpoints": ckpts,
        "lora_files": list_files(LORA_DIR),
        "model_type": "sdxl",
        "supports_face_fix": face_fix_available(),
        "supports_lora_targets": True,
        "supports_refiner": True,
        "supports_face_ref": face_ref_available(),
        "face_ref_methods": [m for m, ok in (("faceid", face_ref_available()), ("pulid", pulid_available()),
                                             ("both", face_ref_available() and pulid_available())) if ok],
        "supports_ref_modes": list(REF_MODES) if ref_available() else ["style", "img2img"],
        "supports_inpaint": True,
        "supports_lora_tags": True,
        "default_checkpoint": DEFAULT_CHECKPOINT,
        "samplers": ["dpmpp_2m", "dpmpp_2m_sde", "dpmpp_sde", "euler", "euler_ancestral", "dpmpp_3m_sde", "uni_pc"],
        "schedulers": ["karras", "normal", "simple", "exponential", "sgm_uniform"],
    }


# ---------------- LoRA loading with a compatibility check ----------------
# ComfyUI silently skips LoRA layers that don't exist in the model, so a LoRA made for another
# model (e.g. SDXL on Qwen) would quietly do nothing. Count what matches and say so.
import comfy.lora
import comfy.lora_convert
import comfy.sd
import folder_paths

_LORA_SUFFIXES = (".lora_up.weight", ".lora_down.weight", ".lora_A.weight", ".lora_B.weight",
                  ".lora_mid.weight", ".lora.up.weight", ".lora.down.weight", ".diff", ".hada_w1_a",
                  ".lokr_w1", ".lokr_w1_a", ".oft_blocks", ".w_norm", ".set_weight")


def apply_lora_checked(model, clip, name, strength_model, strength_clip, model_label):
    """Apply a LoRA like ComfyUI's LoraLoader. Returns (model, clip, warning_or_None).
    Raises ValueError if none of the LoRA's layers match the model."""
    path = folder_paths.get_full_path_or_raise("loras", name)
    lora, metadata = comfy.utils.load_torch_file(path, safe_load=True, return_metadata=True)
    key_map = {}
    if model is not None:
        key_map = comfy.lora.model_lora_keys_unet(model.model, key_map)
    if clip is not None:
        key_map = comfy.lora.model_lora_keys_clip(clip.cond_stage_model, key_map)
    converted = comfy.lora_convert.convert_lora(dict(lora))
    patches = comfy.lora.load_lora(converted, key_map, log_missing=False)
    matched = sum(1 for k in patches if not (isinstance(k, str) and k.endswith(".config")))
    layers = {k[: -len(s)] for k in converted for s in _LORA_SUFFIXES if k.endswith(s)}
    total = len(layers)
    if matched == 0:
        raise ValueError(
            f"LoRA '{name}' doesn't match the {model_label} model (0 of {total or '?'} layers matched). "
            f"It was probably made for a different model — remove it or pick a {model_label} LoRA.")
    warning = None
    if total and matched < 0.5 * total:
        warning = (f"LoRA '{name}': only {matched} of {total} layers matched the {model_label} model, "
                   f"so it may have little effect (probably made for a different model version).")
    print(f"  LoRA {name} @ {strength_model}: {matched}/{total or '?'} layers matched")
    new_model, new_clip = comfy.sd.load_lora_for_models(model, clip, lora, strength_model, strength_clip,
                                                        lora_metadata=metadata)
    return new_model, new_clip, warning


# ---------------- generation (background job, polled by the website) ----------------
class LoraItem(BaseModel):
    name: str
    strength: float = 1.0
    target: str = "all"  # "all" = whole image, "main" = main image only, "face" = face fix only


class GenerateRequest(BaseModel):
    checkpoint: Optional[str] = None
    sampler: str = "dpmpp_2m"
    scheduler: str = "karras"
    steps: int = 25
    cfg: float = 4.5
    width: int = 832
    height: int = 1216
    batch_size: int = 1
    seed: int = -1
    loras: List[LoraItem] = []
    positive: str = ""
    negative: str = ""
    face_fix: bool = False          # redraw faces at full resolution after generating (like ADetailer)
    face_fix_strength: float = 0.4  # how much the face is redrawn: 0.3 subtle ... 0.6 strong
    refiner: bool = False             # second pass: enlarge, then lightly redraw for more detail ("hires fix")
    refiner_checkpoint: Optional[str] = None  # None/"" = same as the main model (keeps LoRAs, fastest)
    refiner_strength: float = 0.3     # how much is redrawn: 0.2 subtle ... 0.5 strong
    refiner_upscale: float = 1.5      # 1.0 = same size, 1.5 = 50% bigger
    refiner_steps: int = 20
    face_ref_images: List[str] = []   # 1-3 pictures of the character's face (data URLs / http addresses)
    face_ref_strength: float = 0.8    # 0.5 light ... 0.8 default ... 1.0+ strong
    face_ref_method: str = "faceid"   # "faceid" (IP-Adapter FaceID Plus v2), "pulid" (PuLID: keeps the face, leaves light / style alone) or "both"
    face_ref_pulid_strength: Optional[float] = None   # "both": PuLID's strength (face_ref_strength is FaceID's)
    # Second picture (works together with the face reference): "pose" (OpenPose ControlNet), "depth" (Depth
    # ControlNet), "style" (IP-Adapter Plus: outfit / light / style) or "img2img" (start from the picture)
    ref_image: Optional[str] = None
    ref_mode: str = "pose"
    ref_strength: float = 0.8         # how strongly it steers (ControlNet / IP-Adapter weight)
    ref_end_percent: float = 0.85     # stop steering at this point of the steps (more freedom for details)
    ref_denoise: float = 0.6          # img2img only: 0.3 close to the picture ... 0.8 loosely based on it
    # Inpaint / outpaint: redraw only where mask_image is white; everything else is pasted back from init_image
    init_image: Optional[str] = None
    mask_image: Optional[str] = None
    inpaint_denoise: float = 1.0      # 1.0 = redraw the masked area fully; 0.5-0.8 = keep some of what's there
    mask_grow: int = 8                # px the mask is widened, so the edge of the redrawn area is covered
    mask_feather: int = 16            # px of soft edge when the result is pasted back


gpu_lock = globals().get("gpu_lock") if isinstance(globals().get("gpu_lock"), type(threading.Lock())) else threading.Lock()
current_job = globals().get("current_job") if isinstance(globals().get("current_job"), dict) else {"id": None}


def progress_hook(value, total, *args, **kwargs):
    job = JOBS.get(current_job["id"])
    if job and total:
        job["step"], job["total_steps"] = int(value), int(total)


comfy.utils.set_progress_bar_global_hook(progress_hook)


def to_data_url(img_tensor):
    arr = (img_tensor.cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    buf = BytesIO()
    Image.fromarray(arr).save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


# ---------------- Face fix (like ADetailer) ----------------
# After an image is made: find each face, cut out the area around it, redraw it at ~1024 px with the
# same model, LoRAs and prompt (only partly, controlled by the strength), and blend it back in.
# Small faces in full-body images get far more detail and look much more like the LoRA's person.
FACE_MODEL_PATH = "/content/ComfyUI/models/ultralytics/face_yolov8m.pt"
VAEEncode = NODE_CLASS_MAPPINGS["VAEEncode"]()
_face_detector = {"model": None}


def face_fix_available():
    if not os.path.exists(FACE_MODEL_PATH):
        return False
    try:
        import ultralytics  # noqa: F401
        return True
    except Exception:
        return False


def detect_faces(pil_img, conf=0.35, max_faces=4):
    """Face boxes (x1, y1, x2, y2), biggest first. Runs on the CPU so it doesn't take GPU memory."""
    if _face_detector["model"] is None:
        from ultralytics import YOLO
        _face_detector["model"] = YOLO(FACE_MODEL_PATH)
    results = _face_detector["model"].predict(pil_img, conf=conf, verbose=False, device="cpu")
    if not results or results[0].boxes is None:
        return []
    boxes = results[0].boxes.xyxy.cpu().numpy().tolist()
    boxes.sort(key=lambda b: (b[2] - b[0]) * (b[3] - b[1]), reverse=True)
    return boxes[:max_faces]


def tensor_to_pil(img_tensor):
    return Image.fromarray((img_tensor.cpu().numpy() * 255).clip(0, 255).astype(np.uint8))


def fix_faces(pil_img, model, positive, negative, vae_m, seed, req, boxes=None):
    """Returns (image, number of faces redrawn)."""
    from PIL import ImageDraw, ImageFilter
    boxes = detect_faces(pil_img) if boxes is None else boxes
    out = pil_img.copy()
    W, H = out.size
    strength = max(0.1, min(float(req.face_fix_strength or 0.4), 0.8))
    fixed = 0
    for x1, y1, x2, y2 in boxes:
        bw, bh = x2 - x1, y2 - y1
        if bw < 12 or bh < 12:
            continue
        # square area around the face with some hair/neck for context, kept inside the image
        side = int(min(max(bw, bh) * 1.8, W, H))
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        left = int(max(0, min(cx - side / 2, W - side)))
        top = int(max(0, min(cy - side / 2, H - side)))
        crop = out.crop((left, top, left + side, top + side))
        work_size = 1024 if side < 1024 else (side // 8) * 8   # SDXL's native size: where the extra detail comes from
        work = crop.resize((work_size, work_size), Image.Resampling.LANCZOS)
        pixels = torch.from_numpy(np.array(work).astype(np.float32) / 255.0)[None, ]
        latent = VAEEncode.encode(vae_m, pixels)[0]
        samples = KSampler.sample(model, seed, req.steps, req.cfg, req.sampler, req.scheduler,
                                  positive, negative, latent, strength)[0]
        redone = tensor_to_pil(VAEDecode.decode(vae_m, samples)[0][0]).resize((side, side), Image.Resampling.LANCZOS)
        # soft-edged mask so the redrawn area blends in without a visible square
        feather = max(4, side // 10)
        mask = Image.new("L", (side, side), 0)
        ImageDraw.Draw(mask).rectangle((feather, feather, side - feather, side - feather), fill=255)
        mask = mask.filter(ImageFilter.GaussianBlur(feather / 2))
        out.paste(redone, (left, top), mask)
        fixed += 1
    return out, fixed


# ---------------- Face reference (IP-Adapter FaceID Plus v2) ----------------
# Keeps a character's face from 1-3 pictures without training: InsightFace reads the face's identity,
# and the adapter steers SDXL toward it (in the main image and in face fix). Comics / films: the same
# reference in every panel or shot keeps the character recognisable.
FACEID_FILES = {
    "ipadapter": os.path.join(MODELS_DIR, "ipadapter", "ip-adapter-faceid-plusv2_sdxl.bin"),
    "lora": os.path.join(MODELS_DIR, "ipadapter", "ip-adapter-faceid-plusv2_sdxl_lora.safetensors"),
    "clip_vision": os.path.join(MODELS_DIR, "clip_vision", "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"),
}
IPADAPTER_NODES = "/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus"
FACEID_LORA_STRENGTH = 0.6  # the adapter's companion LoRA (recommended 0.5-0.7)
_faceid = {}


def face_ref_available():
    if not os.path.isdir(IPADAPTER_NODES) or not all(os.path.exists(p) for p in FACEID_FILES.values()):
        return False
    try:
        import insightface  # noqa: F401
        import onnxruntime  # noqa: F401
        return True
    except Exception:
        return False


def load_faceid():
    """Loads the adapter, image encoder, LoRA and face recogniser once (first use)."""
    if _faceid:
        return _faceid
    import importlib.util
    import comfy.clip_vision
    if "ComfyUI_IPAdapter_plus" not in sys.modules:  # custom nodes aren't auto-loaded when ComfyUI is a library
        spec = importlib.util.spec_from_file_location("ComfyUI_IPAdapter_plus", IPADAPTER_NODES + "/__init__.py",
                                                      submodule_search_locations=[IPADAPTER_NODES])
        module = importlib.util.module_from_spec(spec)
        sys.modules["ComfyUI_IPAdapter_plus"] = module
        spec.loader.exec_module(module)
    ipa = sys.modules["ComfyUI_IPAdapter_plus.IPAdapterPlus"]
    utils = sys.modules["ComfyUI_IPAdapter_plus.utils"]
    print("Loading the face reference models...", flush=True)
    _faceid.update(
        node=ipa.IPAdapterAdvanced(),
        ipadapter=utils.ipadapter_model_loader(FACEID_FILES["ipadapter"]),
        clip_vision=comfy.clip_vision.load(FACEID_FILES["clip_vision"]),
        lora=comfy.utils.load_torch_file(FACEID_FILES["lora"], safe_load=True),
        insightface=utils.insightface_loader("CPU"),  # CPU: no extra GPU memory, fast enough for 1-3 faces
    )
    return _faceid


def load_picture(value):
    """data URL / base64 / http(s) address -> RGB PIL image."""
    value = (value or "").strip()
    if re.match(r"^https?://", value, re.I):
        r = requests.get(value, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        raw = r.content
    else:
        raw = base64.b64decode(re.sub(r"^data:[^,]*,", "", value))
    from PIL import ImageOps
    return ImageOps.exif_transpose(Image.open(BytesIO(raw))).convert("RGB")


def face_ref_tensor(values, size=1024):
    """Up to 3 pictures -> one [N, size, size, 3] batch (each fitted onto a grey square, shape kept)."""
    out = []
    for i, v in enumerate(values[:3]):
        try:
            pil = load_picture(v)
        except Exception as e:
            raise ValueError(f"Face reference {i + 1}: could not read the picture ({e})")
        pil.thumbnail((size, size), Image.Resampling.LANCZOS)
        canvas = Image.new("RGB", (size, size), (127, 127, 127))
        canvas.paste(pil, ((size - pil.width) // 2, (size - pil.height) // 2))
        out.append(torch.from_numpy(np.array(canvas).astype(np.float32) / 255.0))
    return torch.stack(out)


def with_face_ref(model, refs, strength):
    """Model + FaceID LoRA + the face reference (applied after your own LoRAs)."""
    import comfy.model_base
    if not isinstance(model.model, comfy.model_base.SDXL):
        raise ValueError("Face reference only works with SDXL-based checkpoints (SDXL, Pony, Illustrious, ...), "
                         "not with this one — pick an SDXL checkpoint or turn Face reference off")
    F = load_faceid()
    w = max(0.1, min(float(strength), 1.5))
    m = comfy.sd.load_lora_for_models(model, None, F["lora"], FACEID_LORA_STRENGTH, 0)[0]
    try:
        return F["node"].apply_ipadapter(
            m, F["ipadapter"], weight=w, weight_faceidv2=min(w * 1.5, 3.0), weight_type="linear",
            combine_embeds="average", start_at=0.0, end_at=1.0, embeds_scaling="V only", image=refs,
            clip_vision=F["clip_vision"], insightface=F["insightface"])[0]
    except Exception as e:
        if "No face detected" in str(e):
            raise ValueError("No face found in a face reference picture — use clear pictures where the face is "
                             "visible and not too small (a head-and-shoulders crop works best)")
        raise


# ---------------- Face reference: PuLID (the other method, picked on the website) ----------------
# PuLID reads the face with InsightFace (antelopev2) + EVA-CLIP and adds it to SDXL's attention in a way that
# changes the face but leaves the rest of the picture (lighting, style, composition) mostly alone, so it can be
# turned up further than FaceID before the picture looks "off". No LoRA. Its models (~2 GB) download the first
# time PuLID is used in a session.
PULID_NODES = "/content/ComfyUI/custom_nodes/PuLID_ComfyUI"
PULID_FILE = os.path.join(MODELS_DIR, "pulid", "ip-adapter_pulid_sdxl_fp16.safetensors")
ANTELOPE_DIR = os.path.join(MODELS_DIR, "insightface", "models", "antelopev2")
ANTELOPE_FILES = ("1k3d68.onnx", "2d106det.onnx", "genderage.onnx", "glintr100.onnx", "scrfd_10g_bnkps.onnx")
EVA_CLIP = ("QuanSun/EVA-CLIP", "EVA02_CLIP_L_336_psz14_s6B.pt")
FACE_REF_METHODS = ("faceid", "pulid", "both")
_pulid = {}


def pulid_available():
    if not os.path.isdir(PULID_NODES):
        return False
    try:
        import insightface  # noqa: F401
        import onnxruntime  # noqa: F401
        import facexlib  # noqa: F401
        import ftfy  # noqa: F401
        return True
    except Exception:
        return False


def pulid_download(job=None):
    """Downloads PuLID, the antelopev2 face recogniser and EVA-CLIP once (skips what is already there)."""
    from huggingface_hub import hf_hub_download
    missing = not os.path.exists(PULID_FILE) or not all(os.path.exists(os.path.join(ANTELOPE_DIR, f)) for f in ANTELOPE_FILES)
    if missing and job is not None:
        job["phase"] = "downloading PuLID (once per session, ~2 GB)"
    if not os.path.exists(PULID_FILE):
        os.makedirs(os.path.dirname(PULID_FILE), exist_ok=True)
        tmp = hf_hub_download("huchenlei/ipadapter_pulid", os.path.basename(PULID_FILE),
                              local_dir=os.path.join(MODELS_DIR, "pulid", ".hf"))
        os.replace(tmp, PULID_FILE)
    os.makedirs(ANTELOPE_DIR, exist_ok=True)
    for f in ANTELOPE_FILES:
        if not os.path.exists(os.path.join(ANTELOPE_DIR, f)):
            tmp = hf_hub_download("DIAMONIK7777/antelopev2", f, local_dir=os.path.join(ANTELOPE_DIR, ".hf"))
            os.replace(tmp, os.path.join(ANTELOPE_DIR, f))
    hf_hub_download(*EVA_CLIP)   # into the Hugging Face cache, where PuLID's EVA-CLIP loader looks for it


def load_pulid(job=None):
    """Loads PuLID, EVA-CLIP and the face recogniser once (first use)."""
    if _pulid:
        return _pulid
    pulid_download(job)
    if job is not None:
        job["phase"] = "loading PuLID"
    import importlib.util
    if "PuLID_ComfyUI" not in sys.modules:  # custom nodes aren't auto-loaded when ComfyUI is a library
        spec = importlib.util.spec_from_file_location("PuLID_ComfyUI", PULID_NODES + "/__init__.py",
                                                      submodule_search_locations=[PULID_NODES])
        module = importlib.util.module_from_spec(spec)
        sys.modules["PuLID_ComfyUI"] = module
        spec.loader.exec_module(module)
    p = sys.modules["PuLID_ComfyUI.pulid"]
    print("Loading PuLID...", flush=True)
    _pulid.update(
        node=p.ApplyPulid(),
        model=p.PulidModelLoader().load_model(os.path.basename(PULID_FILE))[0],
        eva_clip=p.PulidEvaClipLoader().load_eva_clip()[0],
        face=p.PulidInsightFaceLoader().load_insightface("CPU")[0],   # CPU: no extra GPU memory
    )
    return _pulid


def attn2_patch_count(model):
    patches = model.model_options.get("transformer_options", {}).get("patches_replace", {}).get("attn2", {})
    return sum(len(getattr(p, "callback", [None])) for p in patches.values())


def with_pulid(model, refs, strength, job=None):
    """Model + PuLID face reference ("fidelity" method: closest to the face)."""
    import comfy.model_base
    if not isinstance(model.model, comfy.model_base.SDXL):
        raise ValueError("Face reference only works with SDXL-based checkpoints (SDXL, Pony, Illustrious, ...), "
                         "not with this one — pick an SDXL checkpoint or turn Face reference off")
    P = load_pulid(job)
    w = max(0.05, min(float(strength), 2.0))
    before = attn2_patch_count(model)
    try:
        m = P["node"].apply_pulid(model, P["model"], P["eva_clip"], P["face"], refs, w, 0.0, 1.0, method="fidelity")[0]
    finally:
        P["eva_clip"].to("cpu")   # only needed to read the face: give its GPU memory back to SDXL
        torch.cuda.empty_cache()
    if attn2_patch_count(m) == before:
        raise ValueError("No face found in a face reference picture — use clear pictures where the face is "
                         "visible and not too small (a head-and-shoulders crop works best)")
    return m


def isolate_attn2(model):
    """A clone whose attention patches are its own copies. IP-Adapter and PuLID add a second adapter to the SAME
    patch object as the first one (shared with the model it was cloned from), so without this e.g. the style picture
    would also end up in face fix's model."""
    import copy
    m = model.clone()
    to = dict(m.model_options.get("transformer_options", {}))
    pr = to.get("patches_replace")
    if pr and pr.get("attn2"):
        pr = dict(pr)
        attn2 = {}
        for key, patch in pr["attn2"].items():
            c = copy.copy(patch)
            if hasattr(patch, "callback"):
                c.callback, c.kwargs = list(patch.callback), list(patch.kwargs)
            if hasattr(patch, "multigpu_kwargs"):
                c.multigpu_kwargs = dict(patch.multigpu_kwargs)
            attn2[key] = c
        pr["attn2"] = attn2
        to["patches_replace"] = pr
        m.model_options["transformer_options"] = to
    return m


def apply_face_ref(model, refs, strength, method, job=None, pulid_strength=None):
    """faceid / pulid, or both stacked (FaceID first, then PuLID with its own strength)."""
    if method == "pulid":
        return with_pulid(model, refs, strength, job)
    m = with_face_ref(model, refs, strength)
    if method == "both":
        m = with_pulid(isolate_attn2(m), refs, strength if pulid_strength is None else pulid_strength, job)
    return m


# ---------------- Pose / structure / style picture ----------------
# A second picture next to the face reference. Pose and depth add a ControlNet to the prompt's conditioning;
# style patches the model with IP-Adapter Plus; img2img starts from the picture. Face fix never gets the
# ControlNet (a full-body skeleton must not steer a close-up of the face).
# The ControlNets (2.5 GB each) and IP-Adapter Plus (0.85 GB) download the first time their mode is used.
REF_MODELS = {
    "pose": ("xinsir/controlnet-openpose-sdxl-1.0", "diffusion_pytorch_model.safetensors", "controlnet", "openpose_sdxl_xinsir.safetensors"),
    "depth": ("xinsir/controlnet-depth-sdxl-1.0", "diffusion_pytorch_model.safetensors", "controlnet", "depth_sdxl_xinsir.safetensors"),
    "style": ("h94/IP-Adapter", "sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors", "ipadapter", "ip-adapter-plus_sdxl_vit-h.safetensors"),
}
REF_MODES = ("pose", "depth", "style", "img2img")
_ref = {"mode": None, "model": None, "detectors": {}}


def ref_available():
    try:
        import controlnet_aux  # noqa: F401
        return True
    except Exception:
        return False


def ref_model_path(mode, job=None):
    """Downloads the ControlNet / IP-Adapter for this mode once, returns its path."""
    repo, filename, folder, local_name = REF_MODELS[mode]
    path = os.path.join(MODELS_DIR, folder, local_name)
    if not os.path.exists(path):
        if job is not None:
            job["phase"] = f"downloading the {mode} model (once per session)"
        from huggingface_hub import hf_hub_download
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = hf_hub_download(repo, filename, local_dir=os.path.join(MODELS_DIR, folder, ".hf_" + mode))
        os.replace(tmp, path)
    return path


def load_ref_model(mode, job=None):
    """Keeps one ControlNet / IP-Adapter loaded; switching modes frees the previous one first."""
    if _ref["mode"] == mode and _ref["model"] is not None:
        return _ref["model"]
    _ref.update(mode=None, model=None)
    gc.collect()
    torch.cuda.empty_cache()
    path = ref_model_path(mode, job)
    if job is not None:
        job["phase"] = f"loading the {mode} model"
    if mode == "style":
        load_faceid()   # loads ComfyUI_IPAdapter_plus and the CLIP image encoder it shares
        utils = sys.modules["ComfyUI_IPAdapter_plus.utils"]
        model = utils.ipadapter_model_loader(path)
    else:
        import comfy.controlnet
        model = comfy.controlnet.load_controlnet(path)
    _ref.update(mode=mode, model=model)
    return model


def detector(kind):
    """OpenPose (body, hands, face) or Depth Anything V2 Small. Both run on the CPU: no GPU memory needed."""
    if kind not in _ref["detectors"]:
        if kind == "pose":
            from controlnet_aux import OpenposeDetector
            _ref["detectors"][kind] = OpenposeDetector.from_pretrained("lllyasviel/Annotators")
        else:
            from transformers import pipeline
            _ref["detectors"][kind] = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=-1)
    return _ref["detectors"][kind]


def fit_cover(pil, w, h):
    """Resize to cover w x h and crop the middle, so the map lines up with the image being made (no stretching)."""
    scale = max(w / pil.width, h / pil.height)
    nw, nh = max(w, round(pil.width * scale)), max(h, round(pil.height * scale))
    pil = pil.resize((nw, nh), Image.Resampling.LANCZOS)
    left, top = (nw - w) // 2, (nh - h) // 2
    return pil.crop((left, top, left + w, top + h))


def ref_control_map(pil, mode, w, h):
    """The pose skeleton or depth map for the picture, at the output size."""
    src = fit_cover(pil, w, h)
    if mode == "pose":
        # both resolutions are the length of the SHORT side in controlnet_aux
        out = detector("pose")(src, detect_resolution=min(768, w, h), image_resolution=min(w, h),
                               include_body=True, include_hand=True, include_face=True)
    else:
        out = detector("depth")(src)["depth"].convert("RGB")
    out = out.convert("RGB")
    return out if out.size == (w, h) else out.resize((w, h), Image.Resampling.LANCZOS)


def pil_to_tensor(pil):
    return torch.from_numpy(np.array(pil.convert("RGB")).astype(np.float32) / 255.0)[None, ]


def with_control(positive, negative, control_net, control_image, strength, end_percent, vae_m):
    """Adds a ControlNet to both conditionings (like ComfyUI's Apply ControlNet (Advanced) node)."""
    node = NODE_CLASS_MAPPINGS["ControlNetApplyAdvanced"]()
    try:
        return node.apply_controlnet(positive, negative, control_net, control_image, strength, 0.0, end_percent, vae=vae_m)
    except TypeError:   # older ComfyUI without the vae argument
        return node.apply_controlnet(positive, negative, control_net, control_image, strength, 0.0, end_percent)


def with_style_ref(model, ipadapter, pil, strength, end_percent):
    """IP-Adapter Plus: outfit, light, colours and style from the picture, without fixing where things are."""
    F = load_faceid()
    image = pil_to_tensor(fit_cover(pil, 1024, 1024))
    return F["node"].apply_ipadapter(
        isolate_attn2(model), ipadapter, weight=strength, weight_type="linear", combine_embeds="concat", start_at=0.0,
        end_at=end_percent, embeds_scaling="V only", image=image, clip_vision=F["clip_vision"])[0]


# ---------------- Inpaint / outpaint ----------------
def prepare_mask(mask_pil, size, grow):
    """Black/white mask at the picture's size, widened by `grow` px."""
    from PIL import ImageFilter
    m = mask_pil.convert("L").resize(size, Image.Resampling.NEAREST)
    m = m.point(lambda v: 255 if v > 127 else 0)
    if grow > 0:
        m = m.filter(ImageFilter.GaussianBlur(grow / 2)).point(lambda v: 255 if v > 8 else 0)
    return m


def paste_back(result, original, mask, feather):
    """Only the masked area comes from the result; everything else is the original, pixel for pixel."""
    from PIL import ImageFilter
    if result.size != original.size:
        result = result.resize(original.size, Image.Resampling.LANCZOS)
    soft = mask.filter(ImageFilter.GaussianBlur(feather / 2)) if feather > 0 else mask
    # the soft edge only grows inwards: outside the (grown) mask stays exactly the original
    soft = Image.fromarray(np.minimum(np.array(soft), np.array(mask)))
    return Image.composite(result.convert("RGB"), original.convert("RGB"), soft)


def pil_to_data_url(pil_img):
    buf = BytesIO()
    pil_img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def refine_images(pils, req, model, clip_m, vae_m, positive, negative, seed, job, job_id):
    """Second pass: enlarge each image and lightly redraw it with the refiner model (img2img).
    With a different refiner checkpoint, that checkpoint is loaded in place of the main one."""
    name = (req.refiner_checkpoint or "").strip()
    if name and name != loaded["name"]:
        job["phase"] = f"loading the refiner {name}"
        model, clip_m, vae_m = get_checkpoint(name)
        with torch.inference_mode():
            positive = CLIPTextEncode.encode(clip_m, req.positive)[0]
            negative = CLIPTextEncode.encode(clip_m, req.negative)[0]
    factor = max(1.0, min(float(req.refiner_upscale or 1.0), 2.0))
    strength = max(0.05, min(float(req.refiner_strength or 0.3), 0.8))
    steps = max(4, min(int(req.refiner_steps or 20), 60))
    out = []
    for i, pil in enumerate(pils):
        job["phase"] = f"refining image {i + 1}/{len(pils)}"
        scale = min(factor, 2048 / max(pil.width, pil.height))  # cap the longest side at 2048, keep the shape
        w, h = int(round(pil.width * scale / 8)) * 8, int(round(pil.height * scale / 8)) * 8
        big = pil.resize((w, h), Image.Resampling.LANCZOS) if (w, h) != pil.size else pil
        with torch.inference_mode():
            pixels = torch.from_numpy(np.array(big).astype(np.float32) / 255.0)[None, ]
            latent = VAEEncode.encode(vae_m, pixels)[0]
            samples = KSampler.sample(model, seed + i, steps, req.cfg, req.sampler, req.scheduler,
                                      positive, negative, latent, strength)[0]
            out.append(tensor_to_pil(VAEDecode.decode(vae_m, samples)[0][0]))
        print(f"[{job_id}] refined image {i + 1}: {pil.width}x{pil.height} -> {w}x{h}, strength {strength}"
              + (f", with {name}" if name else ""), flush=True)
    return out


def run_generate(job_id, req):
    job = JOBS[job_id]
    with gpu_lock:
        current_job["id"] = job_id
        job["status"] = "running"
        try:
            model, clip_m, vae_m = get_checkpoint(req.checkpoint)
            available_loras = set(list_files(LORA_DIR))
            batch = max(1, min(int(req.batch_size or 1), 8))
            w = max(256, min(int(req.width), 2048)) // 8 * 8
            h = max(256, min(int(req.height), 2048)) // 8 * 8
            seed = req.seed if req.seed and req.seed > 0 else random.randint(0, 2**63 - 1)
            face_refs = [v for v in (req.face_ref_images or []) if v][:3]
            face_method = (req.face_ref_method or "faceid").lower()
            if face_method not in FACE_REF_METHODS:
                raise ValueError(f"Unknown face reference method: {req.face_ref_method} (use faceid, pulid or both)")
            if face_refs and face_method in ("faceid", "both") and not face_ref_available():
                raise ValueError("Face reference isn't installed in this notebook — open it fresh from GitHub and run cell 1 again")
            if face_refs and face_method in ("pulid", "both") and not pulid_available():
                raise ValueError("PuLID isn't installed in this notebook — open it fresh from GitHub and run cell 1 again")
            if req.face_fix and not face_fix_available():
                raise ValueError("Face fix isn't installed in this notebook — open it fresh from GitHub and run cell 1 again")
            ref_mode = (req.ref_mode or "pose").lower() if req.ref_image else None
            if ref_mode and ref_mode not in REF_MODES:
                raise ValueError(f"Unknown picture mode: {req.ref_mode} (use pose, depth, style or img2img)")
            if ref_mode in ("pose", "depth") and not ref_available():
                raise ValueError("Pose / depth pictures aren't installed in this notebook — open it fresh from GitHub and run cell 1 again")
            ref_strength = max(0.0, min(float(req.ref_strength or 0.8), 1.5))
            ref_end = max(0.1, min(float(req.ref_end_percent or 0.85), 1.0))
            ref_pil = load_picture(req.ref_image) if ref_mode else None
            inpaint = None
            if req.init_image or req.mask_image:
                if not (req.init_image and req.mask_image):
                    raise ValueError("Inpainting needs both the picture and the mask")
                if ref_mode == "img2img":
                    raise ValueError("Img2img and inpainting can't be used together — remove the pose / style picture or set it to another mode")
                init_pil = load_picture(req.init_image)
                w, h = init_pil.width // 8 * 8, init_pil.height // 8 * 8   # the result has the picture's size
                if (w, h) != init_pil.size:
                    init_pil = init_pil.crop((0, 0, w, h))
                grow = max(0, min(int(req.mask_grow or 0), 64))
                inpaint = {"init": init_pil, "mask": prepare_mask(load_picture(req.mask_image), (w, h), grow),
                           "feather": max(0, min(int(req.mask_feather or 0), 128)),
                           "denoise": max(0.05, min(float(req.inpaint_denoise or 1.0), 1.0))}
                if not np.array(inpaint["mask"]).any():
                    raise ValueError("The mask is empty — paint over what should change")
            print(f"[{job_id}] {loaded['name']}, {len(req.loras)} LoRA(s), {req.steps} steps, {w}x{h}, batch={batch}"
                  + (f", face fix {req.face_fix_strength}" if req.face_fix else "")
                  + (f", face reference ({face_method}) x{len(face_refs)} @ {req.face_ref_strength}"
                     + (f" + PuLID @ {req.face_ref_pulid_strength}" if face_method == "both" else "") if face_refs else "")
                  + (f", {ref_mode} picture @ {ref_strength} (until {ref_end:.0%})" if ref_mode and ref_mode != "img2img" else "")
                  + (f", img2img denoise {req.ref_denoise}" if ref_mode == "img2img" else "")
                  + (f", inpaint denoise {inpaint['denoise']} grow {req.mask_grow} feather {inpaint['feather']}" if inpaint else ""))
            warnings = []
            base_model, base_clip = model, clip_m

            def with_loras(items):
                """Base checkpoint + the given LoRAs (model and text encoder)."""
                m, c = base_model, base_clip
                with torch.inference_mode():
                    for l in items:
                        if l.name not in available_loras:
                            raise ValueError(f"LoRA not found on the server: {l.name}")
                        m, c, warning = apply_lora_checked(m, c, l.name, float(l.strength), float(l.strength), "SDXL")
                        if warning and warning not in warnings:
                            warnings.append(warning)
                return m, c

            used = [l for l in req.loras if l.name and l.strength != 0]
            main_items = [l for l in used if l.target in ("all", "main")]
            face_items = [l for l in used if l.target in ("all", "face")]
            if any(l.target == "face" for l in used) and not req.face_fix:
                warnings.append("Face fix is off, so the LoRA(s) set to 'Face fix only' were not used.")
            model, clip_m = with_loras(main_items)
            # Face fix uses its own LoRA set when it differs from the main image's (e.g. a face-only LoRA)
            separate_face = req.face_fix and [(l.name, l.strength) for l in face_items] != [(l.name, l.strength) for l in main_items]
            if separate_face:
                face_model, face_clip = with_loras(face_items)
                print(f"[{job_id}] face fix LoRAs: {', '.join(f'{l.name} @ {l.strength}' for l in face_items) or 'none'}")
            job["warnings"] = warnings
            if face_refs:
                job["phase"] = "reading the face reference"
                with torch.inference_mode():
                    refs = face_ref_tensor(face_refs)
                    model = apply_face_ref(model, refs, req.face_ref_strength, face_method, job, req.face_ref_pulid_strength)
                    if separate_face:
                        face_model = apply_face_ref(face_model, refs, req.face_ref_strength, face_method, job,
                                                    req.face_ref_pulid_strength)
                job["phase"] = None
            ref_map, control = None, None
            base_face_model = model   # LoRAs + face reference, without the style adapter (for face fix)
            if ref_mode in ("pose", "depth", "style"):
                with torch.inference_mode():
                    ref_model = load_ref_model(ref_mode, job)
                    if ref_mode == "style":
                        # only the main image: face fix keeps the face reference but not this
                        model = with_style_ref(model, ref_model, ref_pil, ref_strength, ref_end)
                    else:
                        job["phase"] = "reading the pose" if ref_mode == "pose" else "reading the depth"
                        ref_map = ref_control_map(ref_pil, ref_mode, w, h)
                        control = (ref_model, pil_to_tensor(ref_map))
                job["phase"] = None
            with torch.inference_mode():
                positive = CLIPTextEncode.encode(clip_m, req.positive)[0]
                negative = CLIPTextEncode.encode(clip_m, req.negative)[0]
                # Face fix and the refiner use the prompt WITHOUT the ControlNet
                main_pos, main_neg = positive, negative
                if control:
                    main_pos, main_neg = with_control(positive, negative, control[0], control[1], ref_strength, ref_end, vae_m)
                start_latent = None
                if inpaint:
                    # SDXL redraws only under the mask (noise mask); the rest of the latent is kept
                    start_latent = VAEEncode.encode(vae_m, pil_to_tensor(inpaint["init"]))[0]
                    mask_t = torch.from_numpy(np.array(inpaint["mask"]).astype(np.float32) / 255.0)[None, ]
                    start_latent = NODE_CLASS_MAPPINGS["SetLatentNoiseMask"]().set_mask(start_latent, mask_t)[0]
                    ref_denoise = inpaint["denoise"]
                elif ref_mode == "img2img":
                    start_latent = VAEEncode.encode(vae_m, pil_to_tensor(fit_cover(ref_pil, w, h)))[0]
                    ref_denoise = max(0.05, min(float(req.ref_denoise or 0.6), 1.0))
                if separate_face:
                    face_pos = CLIPTextEncode.encode(face_clip, req.positive)[0]
                    face_neg = CLIPTextEncode.encode(face_clip, req.negative)[0]
                else:
                    face_model, face_pos, face_neg = (base_face_model if ref_mode == "style" else model), positive, negative
                pils, faces_fixed = [], 0
                # one image at a time: lower VRAM use and progress after each image
                for i in range(batch):
                    if start_latent is not None:
                        samples = KSampler.sample(model, seed + i, req.steps, req.cfg, req.sampler, req.scheduler,
                                                  main_pos, main_neg, start_latent, ref_denoise)[0]
                    else:
                        latent = EmptyLatentImage.generate(w, h, 1)[0]
                        samples = KSampler.sample(model, seed + i, req.steps, req.cfg, req.sampler, req.scheduler,
                                                  main_pos, main_neg, latent)[0]
                    image = VAEDecode.decode(vae_m, samples)[0][0]
                    pil = tensor_to_pil(image)
                    if req.face_fix:
                        job["phase"] = "fixing faces"
                        pil, n = fix_faces(pil, face_model, face_pos, face_neg, vae_m, seed + i, req)
                        faces_fixed += n
                        print(f"[{job_id}] image {i + 1}: face fix redrew {n} face(s)")
                        job["phase"] = None
                    if inpaint:   # after face fix, so faces outside the mask can't change either
                        pil = paste_back(pil, inpaint["init"], inpaint["mask"], inpaint["feather"])
                    pils.append(pil)
                    job["done"] = i + 1
            if req.refiner and inpaint:
                warnings.append("The refiner is skipped when inpainting (it would redraw the whole picture).")
            elif req.refiner:
                pils = refine_images(pils, req, model, clip_m, vae_m, positive, negative, seed, job, job_id)
            images = [pil_to_data_url(p) for p in pils]
            job.update(status="done", images=images, seed=seed, width=pils[0].width, height=pils[0].height,
                       faces_fixed=faces_fixed if req.face_fix else None, refined=bool(req.refiner),
                       face_ref=len(face_refs), ref_mode=ref_mode, inpainted=bool(inpaint),
                       ref_map=pil_to_data_url(ref_map) if ref_map is not None else None)
        except Exception as e:
            traceback.print_exc()
            job.update(status="error", error=str(e))
        finally:
            current_job["id"] = None
            gc.collect()
            torch.cuda.empty_cache()


@app.post("/generate")
def generate(req: GenerateRequest):
    batch = max(1, min(int(req.batch_size or 1), 8))
    job_id = new_job("generate", batch_size=batch, done=0, step=0, total_steps=0, images=[])
    threading.Thread(target=run_generate, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "queued", "batch_size": batch}


@app.get("/job/{job_id}")
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        return {"status": "error", "error": last_crash_message() or "Unknown job (the server restarted)"}
    out = {k: v for k, v in job.items() if k != "images"}
    out["job_id"] = job_id
    if job["kind"] == "generate" and job["status"] == "done":
        out["images"] = job["images"]
        out["image"] = job["images"][0] if job["images"] else None
    return out


@app.get("/")
def root():
    return {"status": "ok", "checkpoint": loaded["name"]}


if __name__ == "__main__":
    import uvicorn
    print("Server listening on port 8000.", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

In [ ]:
# Cell 3: start the generation server as a separate program, plus the Cloudflare tunnel.
# If the server crashes (e.g. runs out of memory) only that program dies: it's restarted automatically
# with the same URL, and this notebook keeps running. Re-running this cell restarts the server with the
# current code from cell 2. To stop everything: Runtime → Restart session.
import os, re, sys, time, subprocess, threading, urllib.request
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "requests"], check=True)
SERVER_SCRIPT = "/content/biglust_server.py"
SERVER_LOG = "/content/biglust_server.log"
CRASH_FILE = "/content/server_crash.txt"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"


def server_up():
    try:
        return urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=3).status == 200
    except Exception:
        return False


def log_tail(n=15):
    try:
        return "".join(open(SERVER_LOG, errors="replace").readlines()[-n:])
    except OSError:
        return ""


class Supervisor:
    """Runs the server program and restarts it when it crashes (gives up after 4 crashes in 10 minutes)."""

    def __init__(self):
        self.proc, self.thread, self.stopping, self.gave_up, self.crashes = None, None, False, False, []

    def _loop(self):
        while not self.stopping:
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n===== {time.strftime('%H:%M:%S')} starting the server =====\n")
                log.flush()
                self.proc = subprocess.Popen([sys.executable, "-u", SERVER_SCRIPT], cwd="/content/ComfyUI",
                                             stdout=log, stderr=subprocess.STDOUT,
                                             env={**os.environ, "PYTHONUNBUFFERED": "1"},
                                             start_new_session=True)  # stopping a notebook cell must not stop the server
                code = self.proc.wait()
            if self.stopping:
                break
            why = ("ran out of memory and was stopped by the system" if code in (-9, 137)
                   else f"crashed (exit code {code})")
            self.crashes = [t for t in self.crashes if time.time() - t < 600] + [time.time()]
            if len(self.crashes) >= 4:
                msg = (f"The generation server {why} 4 times in 10 minutes, so it was not restarted. "
                       f"Check the log (cell 4), then run cell 3 again.")
                self.gave_up = True
            else:
                msg = (f"The generation server {why} at {time.strftime('%H:%M')}, so that job was lost. "
                       f"It restarts by itself (models reload in 1-3 minutes). Next time try a smaller size, "
                       f"fewer pictures or batch 1.")
            with open(CRASH_FILE, "w") as fh:
                fh.write(msg)
            with open(SERVER_LOG, "a") as log:
                log.write(f"\n!!!!! {msg}\n")
            if self.gave_up:
                break
            time.sleep(3)

    def start(self):
        self.stopping, self.gave_up = False, False
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        self.stopping = True
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(30)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        if self.thread:
            self.thread.join(35)


# (Re)start the server program
if "SUPERVISOR" in globals():
    print("Restarting the server with the current code...")
    try:
        SUPERVISOR.stop()
    except Exception:
        pass
if os.path.exists(CRASH_FILE):
    os.remove(CRASH_FILE)
SUPERVISOR = Supervisor()
SUPERVISOR.start()

print("Loading the models in the server program (1-3 minutes)...")
t0 = time.time()
while not server_up():
    if SUPERVISOR.gave_up or time.time() - t0 > 900:
        print("The server did not start. Last lines of its log:\n" + log_tail(30))
        raise RuntimeError("Server failed to start — see the log above")
    time.sleep(5)
print(f"Server is up after {time.time() - t0:.0f}s.")

# Cloudflare tunnel (started once; the URL stays the same when the server restarts).
# If it stopped (e.g. the runtime hiccuped), a new one is started — with a NEW URL to paste into the website.
if globals().get("TUNNEL_URL") and (globals().get("TUNNEL_PROC") is None or TUNNEL_PROC.poll() is not None):
    print("The old tunnel stopped, so the old URL no longer works — starting a new one (new URL below).")
    TUNNEL_URL = None
if not globals().get("TUNNEL_URL"):
    subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
    if not os.path.exists(CLOUDFLARED):
        subprocess.run(["wget", "-q", "-O", CLOUDFLARED,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(CLOUDFLARED, 0o755)
    TUNNEL_PROC = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                   start_new_session=True)  # survives stopping a notebook cell
    t0 = time.time()
    while time.time() - t0 < 60 and not globals().get("TUNNEL_URL"):
        line = TUNNEL_PROC.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            TUNNEL_URL = m.group(0)
    threading.Thread(target=lambda: [None for _ in TUNNEL_PROC.stdout], daemon=True).start()  # keep the pipe drained

if globals().get("TUNNEL_URL"):
    print("=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("Paste this into the website's Backend URL field.")
    print("This cell is done; the server keeps running in the background.")
    print("Cell 4 shows its log and memory use. To stop it: Runtime → Restart session.")
    print("=" * 60)
else:
    print("Could not start the Cloudflare tunnel. Run this cell again.")

In [ ]:
# Cell 4 (optional): the server's recent log and memory use. Run it again to refresh.
import subprocess
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"System RAM: {(vm.total - vm.available) / 1e9:.1f} / {vm.total / 1e9:.1f} GB")
except Exception:
    pass
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
    print("No GPU in this runtime (Runtime → Change runtime type → T4 GPU).")
if gpu:
    print(f"GPU memory: {gpu}")
sup = globals().get("SUPERVISOR")
if sup is not None:
    running = sup.proc is not None and sup.proc.poll() is None
    print("Server program:", "running" if running else ("stopped — run cell 3" if sup.gave_up else "starting / restarting"))
    print(f"Crashes in the last 10 minutes: {len([t for t in sup.crashes if __import__('time').time() - t < 600])}")
tp = globals().get("TUNNEL_PROC")
if globals().get("TUNNEL_URL"):
    alive = tp is not None and tp.poll() is None
    print(f"Website URL: {TUNNEL_URL}", "(working)" if alive else "— STOPPED: run cell 3 again and paste its new URL into the website")
print("-" * 60)
try:
    print("".join(open("/content/biglust_server.log", errors="replace").readlines()[-40:]))
except OSError:
    print("No log yet — run cell 3 first.")